Christine's Webscraper

In [34]:
!pip install selenium webdriver-manager beautifulsoup4
!pip install pandas


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 6.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 7.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Check versions
import subprocess
result = subprocess.run(
    ["/Applications/Google Chrome.app/Contents/MacOS/Google Chrome", "--version"],
    capture_output=True, text=True
)
print(result.stdout)

import selenium
print("Selenium version:", selenium.__version__)

Google Chrome 145.0.7632.160 

Selenium version: 4.41.0


In [41]:
# Cell 2: Test driver without ChromeDriverManager
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import pandas as pd
import time
import requests

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)  # No Service() or ChromeDriverManager
driver.get("https://www.google.com")
print("Success! Title:", driver.title)
driver.quit()

Success! Title: Google


<h1>IMDb</h1>

In [ ]:
# Returns imdb reviews url for a given movie name with spaces, e.g. "The Batman"
def get_imdb_reviews_url(driver, movie_name):
    search_query = movie_name.replace(" ", "+")
    search_url = f"https://www.imdb.com/find/?q={search_query}&s=tt&ttype=ft"

    driver.get(search_url)
    time.sleep(5)  # Let the page fully load
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    first_result = soup.find("a", attrs={"class": "ipc-title-link-wrapper"})  # Updated class
    
    if not first_result:
        raise ValueError(f"No IMDB results found for: {movie_name}")
    
    href = first_result["href"]
    movie_id = href.split("/")[2]
    
    movie_url = f"https://www.imdb.com/title/{movie_id}/"
    reviews_url = f"https://www.imdb.com/title/{movie_id}/reviews/"
    return movie_url, reviews_url




# Scrapes movie information including movie title, iMDB rating, year, budget, and gross income across US and Canada
def get_movie_info_imdb(driver, main_url):
    print("Scraping movie info...")
    driver.get(main_url)
    time.sleep(5)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    info = {}

    # Title
    tag = soup.find(attrs={"data-testid": "hero__primary-text"})
    info["title"] = tag.get_text(strip=True) if tag else "N/A"

    # Rating
    tag = soup.find(attrs={"data-testid": "hero-rating-bar__aggregate-rating__score"})
    info["imdb_rating"] = tag.get_text(strip=True).replace("/10", "") if tag else "N/A"

    # Year
    tag = soup.find(attrs={"data-testid": "hero-parent"})
    if tag:
        match = re.search(r'\b(19|20)\d{2}\b', tag.get_text())
        info["year"] = match.group(0) if match else "N/A"
    else:
        info["year"] = "N/A"

    # Budget
    tag = soup.find(attrs={"data-testid": "title-boxoffice-budget"})
    if tag:
        text = tag.get_text(strip=True).replace("Budget", "").replace("(estimated)", "").strip()
        info["budget"] = text
    else:
        info["budget"] = "N/A"

    # Gross US & Canada
    tag = soup.find(attrs={"data-testid": "title-boxoffice-grossdomestic"})
    if tag:
        text = tag.get_text(strip=True).replace("Gross US & Canada", "").strip()
        info["gross_us"] = text
    else:
        info["gross_us"] = "N/A"

    print(f"Movie info: {info}")
    return info





# Scrape a movie's top 50 featured reviews
def scrape_reviews_imdb(driver, reviews_url):
    print("Scraping reviews...")
    driver.get(reviews_url)
    time.sleep(5)

    # Click "Load More" twice to scrape 75 reviews in total then truncate to 50
    for i in range(2):
        try:
            load_more = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button.ipc-see-more__button"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", load_more)
            time.sleep(1)
            driver.execute_script("arguments[0].click();", load_more)
            time.sleep(3)
            print("Loaded more reviews...")
        except:
            print("All reviews loaded.")

    soup = BeautifulSoup(driver.page_source, "html.parser")
    reviews = []
    articles = soup.find_all("article", class_="user-review-item")
    print(f"Found {len(articles)} reviews, parsing...")

    count = 1

    for article in articles:
        review = {}

        review["id"] = f"{count}"
        count += 1

        # Rating — only present if user gave one
        rating_tag = article.find("span", class_="ipc-rating-star--rating")
        review["user_rating"] = rating_tag.get_text(strip=True) if rating_tag else "N/A"

        # Title
        title_tag = article.find("h3", class_="ipc-title__text")
        review["review_title"] = title_tag.get_text(strip=True) if title_tag else "N/A"

        # Review text
        text_tag = article.find("div", attrs={"data-testid": "review-overflow"})
        if not text_tag:
            text_tag = article.find("div", class_=lambda c: c and "content" in c.lower())
        review["review_text"] = text_tag.get_text(strip=True) if text_tag else "N/A"

        # Author
        author_tag = article.find("a", attrs={"data-testid": "author-link"})
        review["author"] = author_tag.get_text(strip=True) if author_tag else "N/A"

        # Date
        date_tag = article.find("li", attrs={"data-testid": "review-date"})
        review["date"] = date_tag.get_text(strip=True) if date_tag else "N/A"

        reviews.append(review)

    reviews = reviews[:50]
    return reviews


In [31]:
def get_imdb_reviews_url_api(movie_name):
    # Use IMDB's suggestion API — fast, no Selenium needed
    search_query = movie_name.replace(" ", "_")
    api_url = f"https://v3.sg.media-imdb.com/suggestion/x/{search_query}.json"
    
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    response = requests.get(api_url, headers=headers)
    data = response.json()
    
    # Filter for movies only (qid == "movie") and grab the first one
    movies = [r for r in data.get("d", []) if r.get("qid") == "movie"]
    
    if not movies:
        raise ValueError(f"No IMDB results found for: {movie_name}")
    
    movie_id = movies[0]["id"]  # e.g. tt6751668
    title = movies[0]["l"]
    
    movie_url = f"https://www.imdb.com/title/{movie_id}/"
    reviews_url = f"https://www.imdb.com/title/{movie_id}/reviews/"
    print(f"Found: {title} → {movie_url}")
    print(f"Found: {title} Reviews → {reviews_url}")
    return movie_url, reviews_url

In [32]:
try:
    driver.quit()
except:
    pass

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

movie_url, reviews_url = get_imdb_reviews_url_api("Parasite")
movie_info = get_movie_info_imdb(driver, movie_url)
reviews = scrape_reviews_imdb(driver, reviews_url)


# movie_url, reviews_url = get_imdb_reviews_url(driver, "Parasite")
# print(movie_url)
# print(reviews_url)

# movie_info = get_movie_info_imdb(driver, movie_url)
# reviews = scrape_reviews_imdb(driver, reviews_url)

driver.quit()

print(f"\nScraped {len(reviews)} reviews total.")

Found: Parasite → https://www.imdb.com/title/tt6751668/
Found: Parasite Reviews → https://www.imdb.com/title/tt6751668/reviews/
Scraping movie info...
Movie info: {'title': 'Parasite', 'imdb_rating': '8.5', 'year': 'N/A', 'budget': '$11,400,000', 'gross_us': '$53,847,897'}
Scraping reviews...
Loaded more reviews...
Loaded more reviews...
Found 74 reviews, parsing...

Scraped 50 reviews total.


In [ ]:
# Save to CSV
filepath = "/Users/christinehan/Documents/GitHub/reviews.csv"
with open(filepath, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["id", "author", "date", "user_rating", "review_title", "review_text"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(reviews)

print(f"Saved to: {filepath}")

<h1>Rotten Tomatoes</h1>

In [42]:
# ---- FUNCTION 1 ----
# Takes a movie name and returns (critics_url, audience_url)
def get_rt_urls(movie_name):
    # Format movie name for RT's URL convention e.g. "The Batman" -> "the_batman"
    slug = movie_name.lower().replace(" ", "_")
    base_url = f"https://www.rottentomatoes.com/m/{slug}"
    critics_url = f"{base_url}/reviews"
    audience_url = f"{base_url}/reviews?type=user"
    print(f"Critics URL: {critics_url}")
    print(f"Audience URL: {audience_url}")
    return critics_url, audience_url




# ---- FUNCTION 2 ----
# Scrapes top 50 critic reviews from RT
def scrape_rt_critic_reviews(driver, critics_url, movie_name):
    print("Scraping RT critic reviews...")
    driver.get(critics_url)
    time.sleep(5)

    dismiss_rt_overlay(driver) 

    reviews = []
    count = 1

    while len(reviews) < 50:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        review_items = soup.find_all("div", attrs={"data-qa": "review-item"})

        for item in review_items:
            if len(reviews) >= 50:
                break

            review = {}
            review["id"] = count
            count += 1

            # Reviewer name
            name_tag = item.find("a", attrs={"data-qa": "review-critic-link"})
            review["reviewer_name"] = name_tag.get_text(strip=True) if name_tag else "N/A"

            # Reviewer username (from href slug)
            if name_tag and name_tag.get("href"):
                review["reviewer_username"] = name_tag["href"].split("/")[-1]
            else:
                review["reviewer_username"] = "N/A"

            # Review date
            date_tag = item.find("span", attrs={"data-qa": "review-date"})
            review["date"] = date_tag.get_text(strip=True) if date_tag else "N/A"

            # Tomato score (Fresh/Rotten)
            score_tag = item.find("score-icon-critic-deprecated") or item.find("span", attrs={"data-qa": "review-score"})
            if score_tag:
                review["tomato_score"] = score_tag.get("sentiment") or score_tag.get_text(strip=True)
            else:
                review["tomato_score"] = "N/A"

            # Review text
            text_tag = item.find("p", attrs={"data-qa": "review-quote"})
            review["review_text"] = text_tag.get_text(strip=True) if text_tag else "N/A"

            reviews.append(review)

        # Try clicking "Load More" if we still need more reviews
        if len(reviews) < 50:
            try:
                load_more = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "rt-button[data-qa='load-more-btn']"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", load_more)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", load_more)
                time.sleep(3)
                print("Loaded more critic reviews...")
            except:
                print("No more critic reviews to load.")
                break

    reviews = reviews[:50]
    df = pd.DataFrame(reviews, columns=["id", "reviewer_name", "reviewer_username", "date", "tomato_score", "review_text"])
    filename = f"{movie_name.replace(' ', '_')}_rt_critic_reviews.csv"
    df.to_csv(filename, index=False)
    print(f"Saved {len(df)} critic reviews to {filename}")
    return df



# ---- FUNCTION 3 ----
# Scrapes 50 most recent audience reviews from RT
def scrape_rt_audience_reviews(driver, audience_url, movie_name):
    print("Scraping RT audience reviews...")
    driver.get(audience_url)
    time.sleep(5)

    reviews = []
    count = 1

    while len(reviews) < 50:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        review_items = soup.find_all("div", attrs={"data-qa": "review-item"})

        for item in review_items:
            if len(reviews) >= 50:
                break

            review = {}
            review["id"] = count
            count += 1

            # Reviewer name
            name_tag = item.find("a", attrs={"data-qa": "review-critic-link"})
            review["reviewer_name"] = name_tag.get_text(strip=True) if name_tag else "N/A"

            # Reviewer username (from href slug)
            if name_tag and name_tag.get("href"):
                review["reviewer_username"] = name_tag["href"].split("/")[-1]
            else:
                review["reviewer_username"] = "N/A"

            # Review date
            date_tag = item.find("span", attrs={"data-qa": "review-date"})
            review["date"] = date_tag.get_text(strip=True) if date_tag else "N/A"

            # Star rating (audience gives 0.5–5 stars)
            rating_tag = item.find("span", attrs={"data-qa": "audience-rating"}) or item.find("score-icon-audience")
            if rating_tag:
                review["rating"] = rating_tag.get("value") or rating_tag.get_text(strip=True)
            else:
                review["rating"] = "N/A"

            # Review text
            text_tag = item.find("p", attrs={"data-qa": "review-quote"})
            review["review_text"] = text_tag.get_text(strip=True) if text_tag else "N/A"

            reviews.append(review)

        # Try clicking "Load More" if we still need more reviews
        if len(reviews) < 50:
            try:
                load_more = WebDriverWait(driver, 5).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "rt-button[data-qa='load-more-btn']"))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", load_more)
                time.sleep(1)
                driver.execute_script("arguments[0].click();", load_more)
                time.sleep(3)
                print("Loaded more audience reviews...")
            except:
                print("No more audience reviews to load.")
                break

    reviews = reviews[:50]
    df = pd.DataFrame(reviews, columns=["id", "reviewer_name", "reviewer_username", "date", "rating", "review_text"])
    filename = f"{movie_name.replace(' ', '_')}_rt_audience_reviews.csv"
    df.to_csv(filename, index=False)
    print(f"Saved {len(df)} audience reviews to {filename}")
    return df





# --------- FUNCTION 4: Dismiss Ads ---------
# RT sometimes shows an interstitial ad before loading reviews, which we need to dismiss to access the reviews
def dismiss_rt_overlay(driver):
    try:
        close_btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-qa='close-overlay-btn']"))
        )
        driver.execute_script("arguments[0].click();", close_btn)
        time.sleep(2)
        print("Dismissed RT overlay.")
    except:
        print("No overlay found, continuing...")

In [44]:
try:
    driver.quit()
except:
    pass

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

movie_name = "Parasite"
critics_url, audience_url = get_rt_urls(movie_name)

driver.get("https://www.rottentomatoes.com/m/parasite/reviews")
time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

# See what data-qa attributes are actually on the page
# for tag in soup.find_all(attrs={"data-qa": True}):
#     print(tag.get("data-qa"), "→", tag.name)

critic_df = scrape_rt_critic_reviews(driver, critics_url, movie_name)
# audience_df = scrape_rt_audience_reviews(driver, audience_url, movie_name)

driver.quit()

Critics URL: https://www.rottentomatoes.com/m/parasite/reviews
Audience URL: https://www.rottentomatoes.com/m/parasite/reviews?type=user
Scraping RT critic reviews...
No overlay found, continuing...
No more critic reviews to load.
Saved 0 critic reviews to Parasite_rt_critic_reviews.csv


In [46]:
try:
    driver.quit()
except:
    pass

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)


driver.get("https://www.rottentomatoes.com/m/parasite/reviews")
time.sleep(5)

# Try to dismiss overlay
try:
    close_btn = WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-qa='close-overlay-btn']"))
    )
    driver.execute_script("arguments[0].click();", close_btn)
    time.sleep(2)
    print("Dismissed overlay")
except:
    print("No overlay found")

# Now see what's on the page
soup = BeautifulSoup(driver.page_source, "html.parser")
for tag in soup.find_all(attrs={"data-qa": True}):
    print(tag.get("data-qa"), "→", tag.name)

No overlay found
close-overlay-btn → action-icon
account-create-username-screen → account-create-username-screen
username-input-label → input-label
username-input → input
continue-btn → rt-button
terms-policies-link → rt-link
privacy-policy-link → rt-link
about-fandango-link → rt-link
login-create-success-screen → account-email-change-success-screen
account-verifying-email-screen → account-verifying-email-screen
retry-link → rt-button
terms-policies-link → rt-link
privacy-policy-link → rt-link
about-fandango-link → rt-link
login-check-email-screen → login-check-email-screen
resend-email-link → rt-button
reset-link → rt-link
terms-policies-link → rt-link
privacy-policy-link → rt-link
about-fandango-link → rt-link
login-error → login-error-screen
header → rt-text
description1 → rt-text
description2 → rt-text
retry-link → rt-link
login-enter-password-screen → login-enter-password-screen
user-email → rt-text
password-input-label → input-label
password-input → input
continue-btn → rt-button